# 08 Business Clustering
Aggregate business features and cluster SMEs into strategic segments.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Aggregate Business Features

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

biz = aggregate_business_features(df)
features = ["followers_count","posts_count","engagement_mean","engagement_rate_mean","view_rate_mean","comment_rate_mean","promo_share","cta_share","location_share","dialect_share","discount_mean","video_share","reel_share"]
X = StandardScaler().fit_transform(biz[features].fillna(0))

## Clustering Experiments

In [4]:
rows, store = [], {}
for k in range(3,9):
    y = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    rows.append({"model":"kmeans","setting":f"k={k}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
    store[("kmeans",f"k={k}")] = y
for k in range(3,9):
    y = GaussianMixture(n_components=k, random_state=42).fit_predict(X)
    rows.append({"model":"gmm","setting":f"components={k}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
    store[("gmm",f"components={k}")] = y
for k in range(3,9):
    for link in ["ward","average","complete"]:
        y = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X)
        rows.append({"model":"hierarchical","setting":f"k={k},linkage={link}","silhouette":silhouette_score(X,y),"clusters":len(np.unique(y))})
        store[("hierarchical",f"k={k},linkage={link}")] = y

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["silhouette","clusters"])
best = exp.iloc[0]
biz["cluster_id"] = store[(best["model"], best["setting"])]
profiles = biz.groupby("cluster_id", as_index=False).agg(
    businesses=("business_name","size"),
    followers_count=("followers_count","mean"),
    engagement_rate_mean=("engagement_rate_mean","mean"),
    view_rate_mean=("view_rate_mean","mean"),
    comment_rate_mean=("comment_rate_mean","mean"),
    promo_share=("promo_share","mean"),
    discount_mean=("discount_mean","mean"),
).sort_values("engagement_rate_mean", ascending=False).reset_index(drop=True)
segments = ["community storytellers","reach-focused brands","discount-driven sellers","passive businesses","conversion-tactical players","video-first builders","niche loyalty brands","emerging challengers"]
profiles["cluster_label"] = [segments[i] if i < len(segments) else f"segment_{i}" for i in range(len(profiles))]
biz = biz.merge(profiles[["cluster_id","cluster_label"]], on="cluster_id", how="left")

  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executabl

## Save Outputs

In [5]:
biz.to_csv(PROCESSED_DIR / "business_clusters.csv", index=False)
profiles.to_csv(PROCESSED_DIR / "business_cluster_profiles.csv", index=False)
exp.to_csv(REPORTS_DIR / "business_clustering_experiments.csv", index=False)
display(exp.head(15))
display(profiles)
print("Insight: business segments enable targeted SME coaching and campaign allocation.")

,model,setting,silhouette,clusters,composite_score
0,hierarchical,"k=8,linkage=average",0.169357,8,1.782153
1,hierarchical,"k=8,linkage=ward",0.160171,8,1.667009
2,hierarchical,"k=8,linkage=complete",0.157465,8,1.633082
3,hierarchical,"k=7,linkage=average",0.173117,7,1.629277
4,hierarchical,"k=6,linkage=complete",0.185876,6,1.589204
5,kmeans,k=8,0.153438,8,1.582608
6,hierarchical,"k=6,linkage=average",0.184405,6,1.570769
7,hierarchical,"k=7,linkage=complete",0.163163,7,1.504514
8,gmm,components=8,0.140894,8,1.425375
9,hierarchical,"k=7,linkage=ward",0.151217,7,1.354773


,cluster_id,businesses,followers_count,engagement_rate_mean,view_rate_mean,comment_rate_mean,promo_share,discount_mean,cluster_label
0,4,2,2021.500000,0.211495,1.465428,0.017832,0.422222,10.486111,community storytellers
1,6,3,6353.000000,0.202130,1.786096,0.016299,0.140216,3.742578,reach-focused brands
2,5,6,14495.000000,0.147413,1.380081,0.012474,0.346247,9.322744,discount-driven sellers
3,3,2,2024.250000,0.146629,1.542789,0.013711,0.299107,6.502976,passive businesses
4,1,5,46163.500000,0.140861,1.239094,0.011769,0.263576,5.466466,conversion-tactical players
5,2,7,2265.142857,0.130770,1.312121,0.011455,0.131566,3.644481,video-first builders
6,7,1,4330.000000,0.122756,1.063322,0.014634,0.363636,9.545455,niche loyalty brands
7,0,14,8022.571429,0.102678,1.072303,0.008518,0.284969,6.924355,emerging challengers


Insight: business segments enable targeted SME coaching and campaign allocation.
